In [22]:
import os
import json
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [23]:
MODEL_PATH = "/kaggle/input/datasets/aditik1234/legalbert/legalbert_contractnli"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=3
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Model loaded successfully")
print("Device:", device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully
Device: cuda


In [24]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(indent + os.path.basename(root) + "/")

    if level < 2:
        for f in files[:10]:
            print(indent + "  " + f)

input/
  datasets/
    aditik1234/
      contract-nli/
        contract-nli/
          raw/
            T_/
              proc_notices/
                notices_020_k/
      legalbert/
        legalbert_contractnli/


In [25]:
import os
import json
import pandas as pd
import numpy as np

DATA_PATH = "/kaggle/input/datasets/aditik1234/contract-nli/contract-nli"

with open(os.path.join(DATA_PATH, "train.json"), "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(os.path.join(DATA_PATH, "dev.json"), "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(os.path.join(DATA_PATH, "test.json"), "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Train documents:", len(train_data["documents"]))
print("Dev documents:", len(dev_data["documents"]))
print("Test documents:", len(test_data["documents"]))

Train documents: 423
Dev documents: 61
Test documents: 123


In [26]:
print("Labels type:", type(train_data["labels"]))

if isinstance(train_data["labels"], list):
    print("Number of labels:", len(train_data["labels"]))
    print("\nFirst label:")
    print(train_data["labels"][0])
else:
    print("Label keys:")
    print(train_data["labels"].keys())

Labels type: <class 'dict'>
Label keys:
dict_keys(['nda-11', 'nda-16', 'nda-15', 'nda-10', 'nda-2', 'nda-1', 'nda-19', 'nda-12', 'nda-20', 'nda-3', 'nda-18', 'nda-7', 'nda-17', 'nda-8', 'nda-13', 'nda-5', 'nda-4'])


In [27]:
label_map = {
    "NotMentioned": 0,
    "Entailment": 1,
    "Contradiction": 2
}

id_to_label = {
    0: "NotMentioned",
    1: "Entailment",
    2: "Contradiction"
}

print(label_map)

{'NotMentioned': 0, 'Entailment': 1, 'Contradiction': 2}


In [28]:
all_data = {
    "train": train_data,
    "dev": dev_data,
    "test": test_data
}

document_map = {}

for split_name, data in all_data.items():
    for document in data["documents"]:
        document_map[str(document["id"])] = {
            "document": document,
            "split": split_name
        }

print("Total documents:", len(document_map))

# Check one document
first_id = list(document_map.keys())[0]
print("Example document ID:", first_id)
print("Text length:", len(document_map[first_id]["document"]["text"]))
print("Number of spans:", len(document_map[first_id]["document"]["spans"]))

Total documents: 607
Example document ID: 34
Text length: 8585
Number of spans: 65


In [29]:
def build_ml_df(data, split_name):
    rows = []

    for document in data["documents"]:
        document_id = str(document["id"])
        document_text = document["text"]

        annotations = document["annotation_sets"][0]["annotations"]

        for label_id, annotation in annotations.items():
            choice = annotation["choice"]
            
            # Get hypothesis from the label definition
            label_info = data["labels"][label_id]
            hypothesis = label_info["hypothesis"]
            short_description = label_info.get("short_description", "")

            rows.append({
                "document_id": document_id,
                "label_id": label_id,
                "short_description": short_description,
                "hypothesis": hypothesis,
                "choice": choice,
                "target": label_map[choice],
                "document_text": document_text,
                "split": split_name
            })

    return pd.DataFrame(rows)


train_df = build_ml_df(train_data, "train")
dev_df = build_ml_df(dev_data, "dev")
test_df = build_ml_df(test_data, "test")

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

Train: (7191, 8)
Dev: (1037, 8)
Test: (2091, 8)


In [30]:
def get_document_spans(document):
    spans = []

    for span_id, (start, end) in enumerate(document["spans"]):
        text = document["text"][start:end].strip()

        if text:
            spans.append({
                "document_id": str(document["id"]),
                "span_id": span_id,
                "text": text
            })

    return spans


all_clauses = []

for document in (
    train_data["documents"]
    + dev_data["documents"]
    + test_data["documents"]
):
    all_clauses.extend(get_document_spans(document))

clauses_df = pd.DataFrame(all_clauses)

print("Total clauses/spans:", len(clauses_df))
print(clauses_df.head())

Total clauses/spans: 47321
  document_id  span_id                                               text
0          34        0       NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT
1          34        1  This NON-DISCLOSURE AND CONFIDENTIALITY AGREEM...
2          34        2  (i) the Office of the United Nations High Comm...
3          34        3  (ii) ________________________ , a company esta...
4          34        4  ________________________ and having its princi...


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

retriever_tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

clause_matrix = retriever_tfidf.fit_transform(
    clauses_df["text"]
)

print("Clause matrix shape:", clause_matrix.shape)

Clause matrix shape: (47321, 30000)


In [32]:
def retrieve_clauses(document_id, hypothesis, top_k=5):
    document_id = str(document_id)

    doc_indices = clauses_df.index[
        clauses_df["document_id"] == document_id
    ].tolist()

    if len(doc_indices) == 0:
        return pd.DataFrame()

    hypothesis_vector = retriever_tfidf.transform(
        [hypothesis]
    )

    scores = cosine_similarity(
        hypothesis_vector,
        clause_matrix[doc_indices]
    ).flatten()

    ranked_positions = np.argsort(scores)[::-1][:top_k]

    selected_indices = [
        doc_indices[i]
        for i in ranked_positions
    ]

    result = clauses_df.loc[selected_indices].copy()

    result["retrieval_score"] = scores[ranked_positions]

    return result.reset_index(drop=True)


# Test retrieval
example = dev_df.iloc[0]

retrieved = retrieve_clauses(
    example["document_id"],
    example["hypothesis"],
    top_k=5
)

print("Hypothesis:")
print(example["hypothesis"])

print("\nRetrieved clauses:")
for _, row in retrieved.iterrows():
    print(
        f"\nSpan {row['span_id']} "
        f"(score={row['retrieval_score']:.4f})"
    )
    print(row["text"])

Hypothesis:
Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.

Retrieved clauses:

Span 39 (score=0.2223)
f) Not to alter, modify, disassemble, reverse engineer or decompile any Confidential Information;

Span 78 (score=0.1421)
<Party>

Span 82 (score=0.1421)
<Party>

Span 56 (score=0.1375)
The Disclosing Party is not obliged to disclose any Confidential Information to the Receiving Party and the Receiving Party shall have the right to refuse to accept any information prior to any disclosure.

Span 22 (score=0.1354)
“Receiving Party” shall mean the Party that receives the Confidential Information directly or indirectly from the Disclosing Party.


In [33]:
def create_clause_pairs(df, top_k=5):
    rows = []

    for _, row in df.iterrows():

        retrieved = retrieve_clauses(
            document_id=row["document_id"],
            hypothesis=row["hypothesis"],
            top_k=top_k
        )

        for _, clause in retrieved.iterrows():

            rows.append({
                "document_id": row["document_id"],
                "label_id": row["label_id"],
                "hypothesis": row["hypothesis"],
                "choice": row["choice"],
                "target": row["target"],
                "span_id": clause["span_id"],
                "clause": clause["text"],
                "retrieval_score": clause["retrieval_score"]
            })

    return pd.DataFrame(rows)


train_pairs = create_clause_pairs(train_df, top_k=5)
dev_pairs = create_clause_pairs(dev_df, top_k=5)
test_pairs = create_clause_pairs(test_df, top_k=5)

print("Train pairs:", train_pairs.shape)
print("Dev pairs:", dev_pairs.shape)
print("Test pairs:", test_pairs.shape)

Train pairs: (35955, 8)
Dev pairs: (5185, 8)
Test pairs: (10455, 8)


In [34]:
def create_pair_text(df):
    return (
        "Hypothesis: "
        + df["hypothesis"].astype(str)
        + " [SEP] Clause: "
        + df["clause"].astype(str)
    )


train_pairs["bert_text"] = create_pair_text(train_pairs)
dev_pairs["bert_text"] = create_pair_text(dev_pairs)
test_pairs["bert_text"] = create_pair_text(test_pairs)

print(train_pairs["bert_text"].iloc[0])

Hypothesis: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information. [SEP] Clause: The Recipient shall not be precluded from disclosing the Confidential Information that is


In [35]:
print("TRAIN")
print(train_pairs["choice"].value_counts())
print()

print("DEV")
print(dev_pairs["choice"].value_counts())
print()

print("TEST")
print(test_pairs["choice"].value_counts())

TRAIN
choice
Entailment       17650
NotMentioned     14100
Contradiction     4205
Name: count, dtype: int64

DEV
choice
Entailment       2595
NotMentioned     2115
Contradiction     475
Name: count, dtype: int64

TEST
choice
Entailment       4840
NotMentioned     4515
Contradiction    1100
Name: count, dtype: int64


In [36]:
sample = train_pairs.sample(
    min(1000, len(train_pairs)),
    random_state=42
)

encoded_sample = tokenizer(
    sample["bert_text"].tolist(),
    truncation=True,
    max_length=512,
    padding=False
)

token_lengths = [
    len(ids)
    for ids in encoded_sample["input_ids"]
]

print("Mean:", np.mean(token_lengths))
print("Median:", np.median(token_lengths))
print("Max:", np.max(token_lengths))
print(">512:", sum(x > 512 for x in token_lengths))

Mean: 56.364
Median: 49.0
Max: 280
>512: 0


In [37]:
def add_evidence_spans(df, data):
    rows = []

    for document in data["documents"]:
        document_id = str(document["id"])
        annotations = document["annotation_sets"][0]["annotations"]

        for label_id, annotation in annotations.items():
            rows.append({
                "document_id": document_id,
                "label_id": label_id,
                "evidence_span_ids": annotation.get("spans", [])
            })

    evidence_df = pd.DataFrame(rows)

    return df.merge(
        evidence_df,
        on=["document_id", "label_id"],
        how="left"
    )


train_pairs = add_evidence_spans(
    train_pairs, train_data
)

dev_pairs = add_evidence_spans(
    dev_pairs, dev_data
)

test_pairs = add_evidence_spans(
    test_pairs, test_data
)

print(train_pairs.columns.tolist())
print(train_pairs.head(2))

['document_id', 'label_id', 'hypothesis', 'choice', 'target', 'span_id', 'clause', 'retrieval_score', 'bert_text', 'evidence_span_ids']
  document_id label_id                                         hypothesis  \
0          34   nda-11  Receiving Party shall not reverse engineer any...   
1          34   nda-11  Receiving Party shall not reverse engineer any...   

         choice  target  span_id  \
0  NotMentioned       0       30   
1  NotMentioned       0       19   

                                              clause  retrieval_score  \
0  The Recipient shall not be precluded from disc...         0.089214   
1  The Recipient shall use the Confidential Infor...         0.049964   

                                           bert_text evidence_span_ids  
0  Hypothesis: Receiving Party shall not reverse ...                []  
1  Hypothesis: Receiving Party shall not reverse ...                []  


In [38]:
def mark_evidence(row):
    gold_spans = row["evidence_span_ids"]

    if not isinstance(gold_spans, list):
        return 0

    return int(row["span_id"] in gold_spans)


train_pairs["is_evidence"] = train_pairs.apply(
    mark_evidence,
    axis=1
)

dev_pairs["is_evidence"] = dev_pairs.apply(
    mark_evidence,
    axis=1
)

test_pairs["is_evidence"] = test_pairs.apply(
    mark_evidence,
    axis=1
)

print(
    train_pairs["is_evidence"].value_counts()
)

print(
    dev_pairs["is_evidence"].value_counts()
)

is_evidence
0    32391
1     3564
Name: count, dtype: int64
is_evidence
0    4705
1     480
Name: count, dtype: int64


In [39]:
def retrieval_evidence_recall(df):
    grouped = df.groupby(
        ["document_id", "label_id"]
    )

    scores = []

    for _, group in grouped:
        gold = set()

        for spans in group["evidence_span_ids"]:
            if isinstance(spans, list):
                gold.update(spans)

        if len(gold) == 0:
            continue

        retrieved = set(
            group.loc[
                group["is_evidence"] == 1,
                "span_id"
            ]
        )

        scores.append(
            len(gold & retrieved) / len(gold)
        )

    return np.mean(scores) if scores else 0


print(
    "Dev evidence recall:",
    retrieval_evidence_recall(dev_pairs)
)

print(
    "Test evidence recall:",
    retrieval_evidence_recall(test_pairs)
)

Dev evidence recall: 0.46639845406132047
Test evidence recall: 0.47491983325316656


In [40]:
train_evidence_pairs = train_pairs[
    train_pairs["is_evidence"] == 1
].copy()

dev_evidence_pairs = dev_pairs[
    dev_pairs["is_evidence"] == 1
].copy()

test_evidence_pairs = test_pairs[
    test_pairs["is_evidence"] == 1
].copy()

print("Evidence pairs:")
print("Train:", len(train_evidence_pairs))
print("Dev:", len(dev_evidence_pairs))
print("Test:", len(test_evidence_pairs))

print("\nClass distribution:")
print(
    train_evidence_pairs["choice"].value_counts()
)

Evidence pairs:
Train: 3564
Dev: 480
Test: 974

Class distribution:
choice
Entailment       3042
Contradiction     522
Name: count, dtype: int64


In [41]:
contradiction_pairs = train_evidence_pairs[
    train_evidence_pairs["choice"] == "Contradiction"
]

print(
    "Contradiction evidence pairs:",
    len(contradiction_pairs)
)

print(
    contradiction_pairs[
        [
            "document_id",
            "label_id",
            "hypothesis",
            "span_id",
            "clause"
        ]
    ].head(10).to_string(index=False)
)

Contradiction evidence pairs: 522
document_id label_id                                                                                                                                 hypothesis  span_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     clause
         86    nda-7 Receiving Party may share some Confidential Information with some third-parties (including consultants, agents and professional advisors).       63                                                                                                                          

In [42]:
def create_retrieved_contexts(df, top_k):
    contexts = []

    for _, row in df.iterrows():

        retrieved = retrieve_clauses(
            document_id=row["document_id"],
            hypothesis=row["hypothesis"],
            top_k=top_k
        )

        if retrieved.empty:
            contexts.append("")
        else:
            contexts.append(
                " ".join(retrieved["text"].tolist())
            )

    return contexts


for k in [1, 3, 5, 10]:

    train_df[f"context_{k}"] = create_retrieved_contexts(
        train_df, k
    )

    dev_df[f"context_{k}"] = create_retrieved_contexts(
        dev_df, k
    )

    test_df[f"context_{k}"] = create_retrieved_contexts(
        test_df, k
    )

    print(f"Finished Top-{k}")

Finished Top-1
Finished Top-3
Finished Top-5
Finished Top-10


In [43]:
for k in [1, 3, 5, 10]:

    train_df[f"bert_text_{k}"] = (
        "Hypothesis: "
        + train_df["hypothesis"].astype(str)
        + " [SEP] Contract Context: "
        + train_df[f"context_{k}"].astype(str)
    )

    dev_df[f"bert_text_{k}"] = (
        "Hypothesis: "
        + dev_df["hypothesis"].astype(str)
        + " [SEP] Contract Context: "
        + dev_df[f"context_{k}"].astype(str)
    )

    test_df[f"bert_text_{k}"] = (
        "Hypothesis: "
        + test_df["hypothesis"].astype(str)
        + " [SEP] Contract Context: "
        + test_df[f"context_{k}"].astype(str)
    )

print(train_df["bert_text_5"].iloc[0])

Hypothesis: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information. [SEP] Contract Context: The Recipient shall not be precluded from disclosing the Confidential Information that is The Recipient shall use the Confidential Information solely for the purpose for which it was disclosed; (i) obtained by the Recipient without restriction from a third party who is not in breach of any obligation as to confidentiality to the owner of such Confidential Information or any other person, or 2.3.2.2 any entity over which the Party exercises effective managerial control; or, 5. All Confidential Information in any form and any medium, including all copies thereof, disclosed to the Recipient shall be returned to UNHCR or destroyed:


In [44]:
for k in [1, 3, 5, 10]:

    sample_texts = train_df[
        f"bert_text_{k}"
    ].sample(
        min(500, len(train_df)),
        random_state=42
    ).tolist()

    encoded = tokenizer(
        sample_texts,
        truncation=True,
        max_length=512,
        padding=False
    )

    lengths = [
        len(x)
        for x in encoded["input_ids"]
    ]

    print(
        f"Top-{k}: "
        f"mean={np.mean(lengths):.1f}, "
        f"median={np.median(lengths):.1f}, "
        f"max={np.max(lengths)}"
    )

Top-1: mean=52.6, median=45.0, max=175
Top-3: mean=122.5, median=117.0, max=354
Top-5: mean=200.7, median=193.0, max=512
Top-10: mean=396.4, median=399.5, max=512


In [45]:
RETRIEVAL_K = 5

X_train = train_df["bert_text_5"].tolist()
X_dev = dev_df["bert_text_5"].tolist()
X_test = test_df["bert_text_5"].tolist()

y_train = train_df["target"].values
y_dev = dev_df["target"].values
y_test = test_df["target"].values

print("Train:", len(X_train))
print("Dev:", len(X_dev))
print("Test:", len(X_test))

Train: 7191
Dev: 1037
Test: 2091
